# Preparación de datos para modelamiento

Este notebook separa la variable objetivo, elimina identificadores, transforma los predictores y divide los datos en entrenamiento y prueba sin utilizar el conjunto de prueba para ajustar el pipeline.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.prepare_features import (
    BINARY_FEATURES, CATEGORICAL_FEATURES, CONTINUOUS_FEATURES,
    IDENTIFIER_COLUMNS, MODEL_FEATURES, TARGET, prepare_model_data
)

## 1. Decisiones de preparación

- Objetivo: `popularity`.
- Se excluyen `track_id`, `artists`, `album_name` y `track_name` para evitar memorización.
- Las variables continuas se estandarizan.
- `track_genre`, `key`, `mode` y `time_signature` se codifican con One-Hot Encoding.
- `explicit` se convierte a 0/1.
- Se utiliza 80% para entrenamiento y 20% para prueba.

In [ ]:
print('Variables continuas:', CONTINUOUS_FEATURES)
print('Variables categóricas:', CATEGORICAL_FEATURES)
print('Variables binarias:', BINARY_FEATURES)
print('Identificadores excluidos:', IDENTIFIER_COLUMNS)

## 2. Ejecutar separación y ajustar el pipeline

In [ ]:
results = prepare_model_data(PROJECT_ROOT)
train_df = results['train']
test_df = results['test']
preprocessor = results['preprocessor']
results['summary']

## 3. Verificar conjuntos resultantes

In [ ]:
print(f'Train: {train_df.shape}')
print(f'Test: {test_df.shape}')
print(f'Nulos en train: {train_df.isna().sum().sum()}')
print(f'Nulos en test: {test_df.isna().sum().sum()}')
print(f'Promedio objetivo train: {train_df[TARGET].mean():.2f}')
print(f'Promedio objetivo test: {test_df[TARGET].mean():.2f}')
train_df.head()

## 4. Comprobar la transformación

In [ ]:
x_train_transformed = preprocessor.transform(train_df[MODEL_FEATURES])
x_test_transformed = preprocessor.transform(test_df[MODEL_FEATURES])
print(f'Train transformado: {x_train_transformed.shape}')
print(f'Test transformado: {x_test_transformed.shape}')
print(f'Variables finales: {len(results["feature_names"])}')

## 5. Prevención de fuga de información

El pipeline se ajusta únicamente con `x_train`. El conjunto `test` se transforma después con los parámetros aprendidos desde entrenamiento. La variable objetivo nunca se utiliza dentro del preprocesador y los identificadores fueron excluidos.